In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\DELL\AppData\Local\Temp\ipykernel_10716\1145051038.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:

# --- PATHS CONFIGURATION (Works in both Jupyter and .py scripts) ---
BASE_DIR = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in globals()
    else os.getcwd()
)
PDF_PATH = os.path.join(BASE_DIR, "Data" , "sample_rag_poc_handbook.pdf")
INDEX_PATH = os.path.join(BASE_DIR, "faiss_index_ollama")

# Local Ollama embedding model
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [4]:




    






def build_poc():
    if not os.path.exists(PDF_PATH):
        raise FileNotFoundError(f"PDF not found at: {PDF_PATH}")

    print("1. Extracting PDF text...")
    loader = PyPDFLoader(PDF_PATH)
    documents = loader.load()

    print("2. Creating chunks...")
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(documents)

    if not chunks:
        raise ValueError("No text extracted! Make sure your PDF is not a scanned image.")

    print(f"3. Building & saving FAISS index ({len(chunks)} chunks)...")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(INDEX_PATH)
    print(f"FAISS index saved successfully at: {INDEX_PATH}\n")


In [5]:


def ask_questions():
    if not os.path.exists(INDEX_PATH):
        raise FileNotFoundError("FAISS index not found. Build the index first.")

    print("Loading FAISS index...")
    vectorstore = FAISS.load_local(
        INDEX_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )

    while True:
        query = input("\nAsk a question (or type 'exit'): ").strip()
        if query.lower() == "exit":
            break

        results = vectorstore.similarity_search(query, k=2)

        print("\n--- Top Matching Context from PDF ---")
        for i, doc in enumerate(results, 1):
            print(f"[Result {i}]: {doc.page_content}\n")


In [6]:

if __name__ == "__main__":
    try:
        build_poc()
        ask_questions()
    except Exception as error:
        print("\nAPPLICATION ERROR: ")
        print(error)

1. Extracting PDF text...
2. Creating chunks...
3. Building & saving FAISS index (13 chunks)...
FAISS index saved successfully at: c:\Users\DELL\PDF_FAISS_RAG_POC\faiss_index_ollama

Loading FAISS index...

--- Top Matching Context from PDF ---
[Result 1]: 1. Company Overview & Leadership
ApexCloud Technologies Pvt. Ltd. was founded in 2019 in Bengaluru, India, by Maya Krishnan (Chief
Executive Officer) and Rohan Deshmukh (Chief Technology Officer). The company provides
enterprise cloud observability, automated incident response, and FinOps cost-optimization software.
As of Q1 2026, ApexCloud employs 420 full-time staff across three regional hubs: Bengaluru (Global
R&D Headquarters), London (EMEA Sales & Support), and Austin, Texas (North America

[Result 2]: ApexCloud Technologies — Internal Operations
& Product Handbook
Document ID: OPS-HB-2026-V4 | Effective Date: January 15, 2026 | Classification: Internal Use Only
Purpose of Document: This handbook serves as a structured synthetic